In [112]:
import os
import io
import re
import json
import time
import zipfile
import requests
import shutil
from pathlib import Path
from typing import Dict, Any, List, Optional
from urllib.parse import quote

from dotenv import load_dotenv

load_dotenv()

True

In [113]:
def authenticate_github() -> Dict[str, str]:
    """
    Returns headers for GitHub REST API.
    Works with public repos even without a token, but a token is strongly recommended
    to avoid rate limits.
    """
    token = os.getenv("GITHUB_TOKEN")

    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "CIROH-AI-Bot-Corpus-Builder"
    }

    if token:
        headers["Authorization"] = f"Bearer {token}"

    return headers

In [114]:
github_headers = authenticate_github()
print("GitHub authentication headers prepared.")
print("Using token:", "Authorization" in github_headers)

GitHub authentication headers prepared.
Using token: True


In [115]:
# =========================
# Configuration
# =========================

GITHUB_OWNER = "CIROH-UA"
BASE_CORPUS_PATH = Path("./github_corpus")

# Repositories to exclude entirely from the GitHub corpus
EXCLUDED_REPOS = {
    "subdomain-hub",
    "ciroh_hub",
    "ciroh_hub_staging",
    "ciroh-portal",
    "ciroh-ua_website",
    "docuhub-staging",
}

OPTIONAL_EXCLUDED_REPOS = {
    ".github",
    "ciroh-ua.github.io",
    "subdomain-ngiab",
    "teehr-may-2023-workshop",
}

# Directories whose contents should never be downloaded into contents/
SKIP_DIR_PREFIXES = {
    ".git/",
    ".github/",
    ".ipynb_checkpoints/",
    "__pycache__/",
    "node_modules/",
    "dist/",
    "build/",
    "_build/",
    "site/",
    ".venv/",
    "venv/",
    "env/",
    ".mypy_cache/",
    ".pytest_cache/",
}

# Exact filenames that are considered useful inputs for all baselines
ALLOWED_EXACT_FILENAMES = {
    "readme",
    "readme.md",
    "readme.rst",
    "readme.txt",
    "license",
    "license.md",
    "license.txt",
    "contributing",
    "contributing.md",
    "contributing.rst",
    "contributing.txt",
    "changelog",
    "changelog.md",
    "changelog.rst",
    "changelog.txt",
    "citation",
    "citation.md",
    "citation.txt",
    "citation.cff",
    "security.md",
    "code_of_conduct.md",
    "requirements.txt",
    "environment.yml",
    "environment.yaml",
    "pyproject.toml",
    "setup.py",
    "setup.cfg",
    "pipfile",
    "pipfile.lock",
}

# Extensions that may be useful, but only under controlled conditions
ALLOWED_TEXT_EXTENSIONS = {
    ".md",
    ".markdown",
    ".rst",
    ".ipynb",
    ".cff",
    ".toml",
    ".yml",
    ".yaml",
    ".py",
}

# Path prefixes that strongly suggest human-readable documentation/tutorial content
ALLOWED_PATH_PREFIXES = {
    "docs/",
    "doc/",
    "documentation/",
    "examples/",
    "example/",
    "tutorials/",
    "tutorial/",
    "notebooks/",
    "notebook/",
    "workshop/",
    "workshops/",
}

# File name substrings that should be excluded even if the extension looks useful
EXCLUDED_NAME_SUBSTRINGS = {
    "checkpoint",
    "dummy",
    "placeholder",
    "deleteme",
    "tmp",
    "temp",
    "backup",
    "~",
}

EXCLUDED_RST_PATH_TOKENS = {
    "/api/",
    "/reference/",
    "/generated/",
    "_build/",
}

# Additional semantic folder signals beyond simple top-level prefixes.
# These help catch useful content in paths like:
# decision_trees/01.script/tutorial.ipynb
# neural_nets/lstm/01.script/example.py
SEMANTIC_PATH_TOKENS = {
    "script",
    "scripts",
    "notebook",
    "notebooks",
    "tutorial",
    "tutorials",
    "example",
    "examples",
    "demo",
    "demos",
    "exercise",
    "exercises",
    "lesson",
    "lessons",
    "lab",
    "labs",
    "training",
    "workflow",
    "workflows",
}

# Extensions allowed when a path contains semantic folder tokens
ALLOWED_SEMANTIC_FOLDER_EXTENSIONS = {
    ".ipynb",
    ".md",
    ".markdown",
    ".rst",
    ".txt",
    ".cff",
}

In [116]:
def github_get(url: str, headers: Dict[str, str], params: Optional[Dict[str, Any]] = None) -> requests.Response:
    resp = requests.get(url, headers=headers, params=params, timeout=60)
    resp.raise_for_status()
    return resp

In [117]:
def list_org_repositories(org_name: str, headers: Dict[str, str]) -> List[Dict[str, Any]]:
    """
    Lists all public repositories for a GitHub organization using pagination.
    """
    repos = []
    page = 1

    while True:
        resp = github_get(
            f"https://api.github.com/orgs/{org_name}/repos",
            headers=headers,
            params={
                "type": "public",
                "per_page": 100,
                "page": page,
                "sort": "full_name",
                "direction": "asc",
            }
        )
        batch = resp.json()
        if not batch:
            break

        repos.extend(batch)
        page += 1

    return repos

In [118]:
def build_repo_manifest_record(repo: Dict[str, Any], contributors_summary: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    license_info = repo.get("license") or {}

    return {
        "id": repo.get("id"),
        "name": repo.get("name"),
        "full_name": repo.get("full_name"),
        "html_url": repo.get("html_url"),
        "description": repo.get("description"),
        "homepage": repo.get("homepage"),
        "default_branch": repo.get("default_branch"),
        "created_at": repo.get("created_at"),
        "updated_at": repo.get("updated_at"),
        "pushed_at": repo.get("pushed_at"),
        "language": repo.get("language"),
        "topics": repo.get("topics", []),
        "fork": repo.get("fork", False),
        "archived": repo.get("archived", False),
        "disabled": repo.get("disabled", False),
        "visibility": repo.get("visibility"),
        "size_kb": repo.get("size"),
        "stargazers_count": repo.get("stargazers_count"),
        "watchers_count": repo.get("watchers_count"),
        "forks_count": repo.get("forks_count"),
        "open_issues_count": repo.get("open_issues_count"),
        "license": {
            "key": license_info.get("key"),
            "name": license_info.get("name"),
            "spdx_id": license_info.get("spdx_id"),
            "url": license_info.get("url"),
        } if license_info else None,
        "contributors_summary": contributors_summary or {
            "contributors_count": 0,
            "top_contributors": []
        }
    }

In [119]:
def get_repo_head_sha(owner: str, repo_name: str, branch: str, headers: Dict[str, str]) -> Optional[str]:
    """
    Returns the HEAD commit SHA for the default branch.
    Returns None when the repository is empty or has no accessible commits.
    """
    url = f"https://api.github.com/repos/{owner}/{repo_name}/commits/{branch}"

    try:
        resp = github_get(url, headers=headers)
        data = resp.json()
        return data["sha"]
    except requests.HTTPError as e:
        status = getattr(e.response, "status_code", None)
        if status == 409:
            print(f"  -> Warning: Repository '{owner}/{repo_name}' is empty or has no accessible commits on '{branch}'.")
            return None
        raise

In [120]:
def get_repo_contributors(owner: str, repo_name: str, headers: Dict[str, str]) -> List[Dict[str, Any]]:
    """
    Retrieves contributors for a repository.
    Returns a simplified list of contributors.
    """
    contributors = []
    page = 1

    while True:
        resp = github_get(
            f"https://api.github.com/repos/{owner}/{repo_name}/contributors",
            headers=headers,
            params={
                "per_page": 100,
                "page": page
            }
        )
        batch = resp.json()

        if not batch:
            break

        for c in batch:
            contributors.append({
                "id": c.get("id"),
                "login": c.get("login"),
                "html_url": c.get("html_url"),
                "type": c.get("type"),
                "contributions": c.get("contributions")
            })

        page += 1

    return contributors


def build_contributors_summary(contributors: List[Dict[str, Any]], top_k: int = 10) -> Dict[str, Any]:
    return {
        "contributors_count": len(contributors),
        "top_contributors": contributors[:top_k]
    }

In [121]:
def download_repo_snapshot_zip(owner: str, repo_name: str, commit_sha: str, headers: Dict[str, str]) -> bytes:
    """
    Downloads a zip archive snapshot for a specific commit SHA using the GitHub API.
    """
    url = f"https://api.github.com/repos/{owner}/{repo_name}/zipball/{commit_sha}"
    resp = github_get(url, headers=headers)
    return resp.content

In [122]:
def normalize_relative_path(relative_path: str) -> str:
    """
    Normalize a repository-relative path without accidentally stripping
    meaningful leading dots such as '.github/'.
    """
    rel = relative_path.replace("\\", "/").strip()

    if rel.startswith("./"):
        rel = rel[2:]

    return rel


def should_skip_by_path(relative_path: str) -> bool:
    rel = normalize_relative_path(relative_path)
    rel_lower = rel.lower()

    for prefix in SKIP_DIR_PREFIXES:
        if rel_lower.startswith(prefix):
            return True

    if "/.ipynb_checkpoints/" in rel_lower:
        return True

    return False


def normalize_path_segment(segment: str) -> str:
    """
    Normalize a path segment so folders like:
      01.script
      02_input
      03-output
    become semantically searchable as:
      script
      input
      output
    """
    s = segment.lower().strip()

    # remove leading numbering like 01., 02_, 03-, 04
    s = re.sub(r"^\d+[\._\- ]*", "", s)

    # replace separators with spaces
    s = re.sub(r"[\._\-]+", " ", s)

    # collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s


def path_has_semantic_folder_signal(relative_path: str) -> bool:
    """
    Returns True when any directory segment in the path appears semantically useful
    for tutorial/example/workflow content, even if the structure is not captured
    by ALLOWED_PATH_PREFIXES.
    """
    rel = normalize_relative_path(relative_path)
    parts = Path(rel).parts

    # check only directory segments, not the filename
    dir_parts = parts[:-1]

    for seg in dir_parts:
        norm = normalize_path_segment(seg)
        if not norm:
            continue

        tokens = set(norm.split())
        if tokens & SEMANTIC_PATH_TOKENS:
            return True

    return False


def classify_selection_reason(relative_path: str, repo_name: str) -> Optional[str]:
    """
    Returns a non-null reason if the file should be downloaded into contents/.
    Otherwise returns None.
    """
    rel = normalize_relative_path(relative_path)
    rel_lower = rel.lower()
    repo_name_lower = (repo_name or "").strip().lower()

    file_name = Path(rel_lower).name
    suffix = Path(file_name).suffix.lower()

    if should_skip_by_path(rel_lower):
        return None

    if any(token in file_name for token in EXCLUDED_NAME_SUBSTRINGS):
        return None

    # Exclude auto-generated / reference-style .rst files in noisy paths
    if suffix == ".rst" and any(token in rel_lower for token in EXCLUDED_RST_PATH_TOKENS):
        return None

    # Exclude repo autodoc stubs like hydrotools.something.rst
    if suffix == ".rst" and repo_name_lower and file_name.startswith(f"{repo_name_lower}."):
        return None

    # Exclude LICENSE.txt when it appears in build/vendor-like paths
    if file_name == "license.txt" and (
        "_build/" in rel_lower
        or "/vendor/" in rel_lower
        or "/vendors/" in rel_lower
        or "/third_party/" in rel_lower
        or "/third-party/" in rel_lower
        or "/site-packages/" in rel_lower
        or "/dist/" in rel_lower
        or "/build/" in rel_lower
    ):
        return None

    # 1. Always include high-value exact filenames
    if file_name in ALLOWED_EXACT_FILENAMES:
        return "allowed_exact_filename"

    # 2. Include documentation/tutorial/example paths only for selected useful extensions
    if any(rel_lower.startswith(prefix) for prefix in ALLOWED_PATH_PREFIXES):
        if suffix in ALLOWED_TEXT_EXTENSIONS:
            return "allowed_path_prefix"

    # 3. Include files located in semantically meaningful folders,
    #    but only for notebook/document-like files
    if path_has_semantic_folder_signal(rel_lower):
        if suffix in ALLOWED_SEMANTIC_FOLDER_EXTENSIONS:
            return "allowed_semantic_folder"

    # 4. Include top-level markdown/rst/cff docs
    if "/" not in rel_lower and suffix in {".md", ".markdown", ".rst", ".cff"}:
        return "allowed_top_level_doc"

    # 5. Include top-level notebooks
    if "/" not in rel_lower and suffix == ".ipynb":
        return "allowed_top_level_notebook"

    # 6. Do not include notebooks elsewhere unless already captured by a path rule above
    if suffix == ".ipynb":
        return None

    # 7. Include environment/build files only if explicitly whitelisted by exact filename
    if suffix in {".toml", ".yml", ".yaml", ".py"}:
        return None

    # 8. Never generally include .txt by extension alone
    if suffix == ".txt":
        return None

    return None

In [123]:
def extract_selected_files_from_zip(zip_bytes: bytes, output_dir: Path, repo_name: str) -> Dict[str, Any]:
    """
    Builds files_manifest.json for all files in the repository and extracts only
    selected files into contents/.
    Cleans previous extracted contents to avoid stale files from earlier runs.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    contents_dir = output_dir / "contents"

    # Clean previous extracted contents so the corpus reflects only the current selection policy
    if contents_dir.exists():
        shutil.rmtree(contents_dir)
    contents_dir.mkdir(parents=True, exist_ok=True)

    files_manifest = []

    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        members = [m for m in zf.infolist() if not m.is_dir()]

        for member in members:
            parts = Path(member.filename).parts
            if len(parts) < 2:
                continue

            relative_path = str(Path(*parts[1:]))
            relative_path_norm = relative_path.replace("\\", "/")

            selection_reason = classify_selection_reason(relative_path_norm, repo_name=repo_name)
            downloaded = selection_reason is not None

            if downloaded:
                target_path = contents_dir / relative_path_norm
                target_path.parent.mkdir(parents=True, exist_ok=True)

                with zf.open(member, "r") as src, open(target_path, "wb") as dst:
                    dst.write(src.read())

            files_manifest.append({
                "path": relative_path_norm,
                "file_name": Path(relative_path_norm).name,
                "extension": Path(relative_path_norm).suffix.lower(),
                "size_bytes": member.file_size,
                "downloaded": downloaded,
                "selection_reason": selection_reason,
            })

    downloaded_files_count = sum(1 for x in files_manifest if x["downloaded"])

    return {
        "files_manifest": files_manifest,
        "downloaded_files_count": downloaded_files_count,
    }

In [124]:
def download_github_corpus(
    repos: List[Dict[str, Any]],
    headers: Dict[str, str],
    base_path: Path = BASE_CORPUS_PATH
) -> Dict[str, Any]:
    """
    Downloads the GitHub Level-0 corpus for selected repositories.
    Saves:
      - repo_metadata.json
      - archive_info.json
      - files_manifest.json
      - contributors.json
      - contents/ (selected files only)
    """
    base_path.mkdir(parents=True, exist_ok=True)

    success_count = 0
    failed_repos = []
    skipped_repos = []
    report = []
    manifest_records = []

    for index, repo in enumerate(repos, start=1):
        repo_name = repo["name"]
        owner = repo["owner"]["login"]
        default_branch = repo["default_branch"]

        print(f"[{index}/{len(repos)}] Processing repository: {owner}/{repo_name}")

        repo_dir = base_path / repo_name
        repo_dir.mkdir(parents=True, exist_ok=True)

        try:
            head_sha = get_repo_head_sha(owner, repo_name, default_branch, headers)

            if not head_sha:
                print(f"  -> Skipped: empty repository or no accessible HEAD commit")
                skipped_repos.append(repo_name)
                report.append({
                    "repo_name": repo_name,
                    "full_name": repo.get("full_name"),
                    "status": "skipped",
                    "reason": "empty_repository"
                })
                continue

            contributors = get_repo_contributors(owner, repo_name, headers)
            contributors_summary = build_contributors_summary(contributors)

            metadata = build_repo_manifest_record(repo, contributors_summary=contributors_summary)

            archive_info = {
                "owner": owner,
                "repo_name": repo_name,
                "default_branch": default_branch,
                "frozen_commit_sha": head_sha,
                "downloaded_at_epoch": time.time(),
                "archive_format": "zip",
            }

            zip_bytes = download_repo_snapshot_zip(owner, repo_name, head_sha, headers)
            extracted = extract_selected_files_from_zip(zip_bytes, repo_dir, repo_name=repo_name)

            with open(repo_dir / "repo_metadata.json", "w", encoding="utf-8") as f:
                json.dump(metadata, f, indent=4, ensure_ascii=False)

            with open(repo_dir / "archive_info.json", "w", encoding="utf-8") as f:
                json.dump(archive_info, f, indent=4, ensure_ascii=False)

            with open(repo_dir / "files_manifest.json", "w", encoding="utf-8") as f:
                json.dump(extracted["files_manifest"], f, indent=4, ensure_ascii=False)

            with open(repo_dir / "contributors.json", "w", encoding="utf-8") as f:
                json.dump(contributors, f, indent=4, ensure_ascii=False)

            report.append({
                "repo_name": repo_name,
                "full_name": repo["full_name"],
                "frozen_commit_sha": head_sha,
                "total_files": len(extracted["files_manifest"]),
                "downloaded_files": extracted["downloaded_files_count"],
                "contributors_count": len(contributors),
                "status": "success",
            })

            manifest_records.append(metadata)
            success_count += 1

        except Exception as e:
            print(f"  -> Failure: {e}")
            failed_repos.append(repo_name)
            report.append({
                "repo_name": repo_name,
                "full_name": repo.get("full_name"),
                "status": "failed",
                "error": str(e)
            })

    with open(base_path / "repo_download_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=4, ensure_ascii=False)

    with open(base_path / "github_org_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest_records, f, indent=4, ensure_ascii=False)

    return {
        "total_processed": len(repos),
        "successful_downloads": success_count,
        "failed_downloads": len(failed_repos),
        "skipped_downloads": len(skipped_repos),
        "failed_repos": failed_repos,
        "skipped_repos": skipped_repos,
    }

In [125]:
all_repos = list_org_repositories(GITHUB_OWNER, github_headers)
print(f"Repositories found in organization: {len(all_repos)}")

selected_repos = []
for repo in all_repos:
    name = repo.get("name")
    if name in EXCLUDED_REPOS:
        continue
    if name in OPTIONAL_EXCLUDED_REPOS:
        continue
    selected_repos.append(repo)

print(f"Repositories selected for download: {len(selected_repos)}")
print("Sample selected repos:", [r["name"] for r in selected_repos[:10]])

Repositories found in organization: 61
Repositories selected for download: 51
Sample selected repos: ['api-nwm-gcp', 'awi-ciroh-image', 'CAMELS_data_sample', 'cfe_v1.0', 'CIROH-open-source-project-template', 'ciroh_pyngiab', 'Community-Streamflow-Evaluation-System', 'community_hf_patcher', 'Conferences', 'datastreamcli']


In [126]:
report = download_github_corpus(
    repos=selected_repos,
    headers=github_headers,
    base_path=BASE_CORPUS_PATH
)

print("\n" + "=" * 50)
print("GITHUB CORPUS ACQUISITION REPORT")
print("=" * 50)
print(f"Total processed: {report['total_processed']}")
print(f"Successful downloads: {report['successful_downloads']}")
print(f"Skipped downloads: {report['skipped_downloads']}")
print(f"Failed downloads: {report['failed_downloads']}")

if report["skipped_repos"]:
    print("\nSkipped repositories:")
    for repo_name in report["skipped_repos"]:
        print(" -", repo_name)

if report["failed_repos"]:
    print("\nFailed repositories:")
    for repo_name in report["failed_repos"]:
        print(" -", repo_name)

[1/51] Processing repository: CIROH-UA/api-nwm-gcp
[2/51] Processing repository: CIROH-UA/awi-ciroh-image
[3/51] Processing repository: CIROH-UA/CAMELS_data_sample
[4/51] Processing repository: CIROH-UA/cfe_v1.0
[5/51] Processing repository: CIROH-UA/CIROH-open-source-project-template
[6/51] Processing repository: CIROH-UA/ciroh_pyngiab
[7/51] Processing repository: CIROH-UA/Community-Streamflow-Evaluation-System
[8/51] Processing repository: CIROH-UA/community_hf_patcher
[9/51] Processing repository: CIROH-UA/Conferences
[10/51] Processing repository: CIROH-UA/datastreamcli
[11/51] Processing repository: CIROH-UA/data_access_example
[12/51] Processing repository: CIROH-UA/deep_bucket_lab
[13/51] Processing repository: CIROH-UA/DEVCON_SNOW_ML
[14/51] Processing repository: CIROH-UA/forcingprocessor
[15/51] Processing repository: CIROH-UA/forecast_data_overlay
[16/51] Processing repository: CIROH-UA/GEE_Workshop
[17/51] Processing repository: CIROH-UA/hf_pmtiles
[18/51] Processing repos

In [127]:
from pathlib import Path
import json
from collections import defaultdict

OLD_CORPUS_DIR = Path("./github_corpus_v5")
NEW_CORPUS_DIR = Path("./github_corpus")


def load_downloaded_paths_by_repo(base_dir: Path) -> dict[str, set[str]]:
    """
    Reads files_manifest.json from each repo directory and returns:
        {repo_name: {downloaded_relative_paths}}
    considering only files with downloaded == True.
    """
    by_repo = {}

    if not base_dir.exists():
        raise FileNotFoundError(f"Directory not found: {base_dir}")

    repo_dirs = sorted([p for p in base_dir.iterdir() if p.is_dir()], key=lambda p: p.name.lower())

    for repo_dir in repo_dirs:
        manifest_path = repo_dir / "files_manifest.json"
        if not manifest_path.exists():
            by_repo[repo_dir.name] = set()
            continue

        with open(manifest_path, "r", encoding="utf-8") as f:
            manifest = json.load(f)

        downloaded_paths = set()
        for rec in manifest:
            if isinstance(rec, dict) and rec.get("downloaded", False):
                path_value = rec.get("path")
                if path_value:
                    downloaded_paths.add(str(path_value).replace("\\", "/"))

        by_repo[repo_dir.name] = downloaded_paths

    return by_repo


old_by_repo = load_downloaded_paths_by_repo(OLD_CORPUS_DIR)
new_by_repo = load_downloaded_paths_by_repo(NEW_CORPUS_DIR)

all_repos = sorted(set(old_by_repo.keys()) | set(new_by_repo.keys()), key=str.lower)

removed_by_repo = {}
added_by_repo = {}

all_removed = []
all_added = []

for repo_name in all_repos:
    old_paths = old_by_repo.get(repo_name, set())
    new_paths = new_by_repo.get(repo_name, set())

    removed = sorted(old_paths - new_paths)
    added = sorted(new_paths - old_paths)

    removed_by_repo[repo_name] = removed
    added_by_repo[repo_name] = added

    for p in removed:
        all_removed.append((repo_name, p))
    for p in added:
        all_added.append((repo_name, p))

print("=" * 70)
print("GLOBAL COMPARISON OF DOWNLOADED FILES")
print("=" * 70)
print(f"Repositories in old corpus: {len(old_by_repo)}")
print(f"Repositories in new corpus: {len(new_by_repo)}")
print(f"Union of repository names: {len(all_repos)}")
print()
print(f"Files downloaded before but NOT now: {len(all_removed)}")
print(f"Files downloaded now but NOT before: {len(all_added)}")

repos_with_removed = [r for r in all_repos if removed_by_repo[r]]
repos_with_added = [r for r in all_repos if added_by_repo[r]]

print(f"Repositories with removed files: {len(repos_with_removed)}")
print(f"Repositories with added files: {len(repos_with_added)}")

print("\n" + "=" * 70)
print("FILES REMOVED IN NEW CORPUS (present in v5, absent now)")
print("=" * 70)
if all_removed:
    for repo_name, path_value in all_removed:
        print(f"{repo_name}: {path_value}")
else:
    print("None")

print("\n" + "=" * 70)
print("FILES ADDED IN NEW CORPUS (absent in v5, present now)")
print("=" * 70)
if all_added:
    for repo_name, path_value in all_added:
        print(f"{repo_name}: {path_value}")
else:
    print("None")

print("\n" + "=" * 70)
print("PER-REPOSITORY SUMMARY")
print("=" * 70)
for repo_name in all_repos:
    n_removed = len(removed_by_repo[repo_name])
    n_added = len(added_by_repo[repo_name])
    if n_removed or n_added:
        print(f"{repo_name}: -{n_removed} / +{n_added}")

# Optional quick spotlight for hydromachine-tutorials
target_repo = "hydromachine-tutorials"
if target_repo in all_repos:
    print("\n" + "=" * 70)
    print(f"DETAIL FOR {target_repo}")
    print("=" * 70)
    print("Removed:")
    if removed_by_repo[target_repo]:
        for p in removed_by_repo[target_repo]:
            print("  -", p)
    else:
        print("  None")

    print("Added:")
    if added_by_repo[target_repo]:
        for p in added_by_repo[target_repo]:
            print("  +", p)
    else:
        print("  None")

GLOBAL COMPARISON OF DOWNLOADED FILES
Repositories in old corpus: 51
Repositories in new corpus: 51
Union of repository names: 51

Files downloaded before but NOT now: 32
Files downloaded now but NOT before: 81
Repositories with removed files: 4
Repositories with added files: 14

FILES REMOVED IN NEW CORPUS (present in v5, absent now)
hydrotools: docs/hydrotools._restclient._iterable_nonstring.rst
hydrotools: docs/hydrotools._restclient._restclient.rst
hydrotools: docs/hydrotools._restclient._restclient_sigs.rst
hydrotools: docs/hydrotools._restclient.async_client.rst
hydrotools: docs/hydrotools._restclient.async_helpers.rst
hydrotools: docs/hydrotools._restclient.rst
hydrotools: docs/hydrotools._restclient.urllib.rst
hydrotools: docs/hydrotools._restclient.urllib_types.rst
hydrotools: docs/hydrotools._restclient.utilities.rst
hydrotools: docs/hydrotools.caches.hdf.rst
hydrotools: docs/hydrotools.caches.rst
hydrotools: docs/hydrotools.events.event_detection.decomposition.rst
hydrotools